In [7]:
from pprint import pprint

from magician.agent.graph import run_agent

In [2]:
question = "Как изменились общие клиентские средства и чистая прибыль Столичного банка за последний квартал?"
answer = await run_agent(question)

→ Анализирую вопрос…
→ Изучаю доступные таблицы и показатели…
→ Проверяю значения dim_bank.bank_name…
→ Выполняю запрос к данным…
→ Выполняю запрос к данным…
✓ Ответ готов


In [11]:
question = "Как изменились общие клиентские средства и чистая прибыль частных банков за последний квартал?"
answer = await run_agent(question)

→ Анализирую вопрос…
→ Изучаю доступные таблицы и показатели…
→ Проверяю значения dim_bank.ownership_type…
→ Проверяю значения dim_bank.ownership_type…
→ Выполняю запрос к данным…
✓ Ответ готов


In [12]:
print(answer.answer)
print("\nПлан:", answer.calculation_plan or "—")
print("\nSQL:", answer.sql or "—")

За последний доступный квартал (октябрь–декабрь 2025 года) общие клиентские средства частных банков выросли с 218 724,68 до 225 074,73 млн руб.: на 6 350,05 млн руб. (+2,90%). Надёжно определить изменение чистой прибыли нельзя: в текущем квартале отсутствует одна строка доходов частного банка (23 из ожидаемых 24 банк-месяцев), а отсутствующую строку нельзя считать нулевой. Для ориентира по имеющимся строкам: прибыль составила 3 111,37 млн руб. против 3 068,29 млн руб. в предыдущем квартале, но это сравнение неполное.

План: Максимальный доступный месяц — декабрь 2025. Клиентские средства сравнивались как снимки на 30.09 и 31.12.2025 после суммирования трёх непересекающихся сегментов; чистая прибыль — как сумма месячных потоков за октябрь–декабрь 2025 против июля–сентября 2025, с проверкой полноты.

SQL: WITH
private_banks AS (
    SELECT bank_id
    FROM dim_bank
    WHERE ownership_type = 'private'
),
max_dates AS (
    SELECT
        (SELECT MAX(report_month) FROM fact_customers_mont